# nAChR Variant Effect Predictor — MASTER Notebook

**Goal:** Predict whether a single-amino-acid substitution in nicotinic acetylcholine receptors (nAChRs) causes **loss-of-function (LOF)** or **gain-of-function (GOF)**.

**Approach:** Compare **domain-driven** (engineered features from AAIndex, PDB structures, BLOSUM62, Grantham distance) vs **data-driven** (ordinal, one-hot, full-sequence) encodings across 8 ML models with nested cross-validation.

## Pipeline Overview
```
Data Loading → Feature Encoding → Nested CV → Evaluation → Species Transfer → Ablation
      │               │                │             │               │            │
  human+mouse   6 strategies      8 models     SHAP+stats    human/mouse    per-feature
  (+rat ready)  (52 features max)  Optuna HP    plots        3 conditions   +groups
```

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure vep_nachr is importable
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── VEP-nAChR modules ──────────────────────────────────────
from vep_nachr.config import (
    NACHR_SUBUNITS, PDB_MAPPING, FEATURE_GROUPS,
    ENCODING_STRATEGIES, CORE_MODELS, ALL_MODELS,
    DOMAIN_DRIVEN_ENCODINGS, DATA_DRIVEN_ENCODINGS,
    RESULTS_DIR, Config, get_config, FEATURE_CACHE_DIR,
)
from vep_nachr.data.loader import load_dataset, load_multi_species_dataset
from vep_nachr.data.encoders import OrdinalEncoder, OneHotEncoder, FullSequenceEncoder, get_encoder
from vep_nachr.features.encoder import NachrFeatureEncoder
from vep_nachr.features.noah_original import NoahOriginalFeatureEncoder, CombinedFeatureEncoder
from vep_nachr.models.registry import AVAILABLE_MODELS, get_model
from vep_nachr.training.cross_validation import (
    nested_cross_validation,
    simple_cross_validation,
    species_transfer_cv,
    SpeciesTransferResult,
    ExperimentResult,
)
from vep_nachr.training.evaluation import (
    compute_metrics,
    compute_per_class_metrics,
    aggregate_results,
    paired_comparison_test,
    FeatureImportanceAnalyzer,
    plot_feature_importance,
    plot_shap_summary,
    plot_species_transfer_comparison,
    plot_comparison_heatmap,
)

print(f"VEP-nAChR ready — {len(NACHR_SUBUNITS)} subunits, {len(AVAILABLE_MODELS)} models available")
print(f"Encodings: {list(ENCODING_STRATEGIES.keys())}")

## 2. Data Loading

The primary data source is `nachr_db_cleaned.xlsx` — a manually curated database of nAChR mutations with experimentally validated LOF/GOF effects.

Multi-species support is built in: pass `--mouse-data` / `--rat-data` to the CLI runner, or use `load_multi_species_dataset()` here.

In [ ]:
# Load human data (default)
df, labels, sequences = load_dataset()

print(f"Mutations: {len(df)}")
print(f"Subunits:  {sorted(df['subunit'].unique())}")
print(f"LOF: {(labels==0).sum()} | GOF: {(labels==1).sum()}")
print(f"Sequences: {len(sequences)} subunits")
print(f"\nLabel distribution:\n{pd.Series(labels).value_counts().to_string()}")
print(f"\nSubunit distribution:\n{df['subunit'].value_counts().to_string()}")

In [ ]:
# Multi-species loading (when mouse/rat data is available)
# Uncomment when your data is ready:
# df_multi, labels_multi, seqs_multi = load_multi_species_dataset(
#     human_path=None,  # uses default
#     mouse_path='path/to/mouse_data.xlsx',
#     # rat_path='path/to/rat_data.xlsx',  # optional
# )
# print(f"Total: {len(df_multi)} mutations")
# print(df_multi['species'].value_counts())

## 3. Encoding Strategies

Six encoding strategies are available. The key comparison is **domain-driven** (expert knowledge) vs **data-driven** (raw sequence).

| Encoding | Type | Features | Description |
|----------|------|----------|-------------|
| **engineered** | Domain | 52 | 8 AAIndex × 3 (wt/mt/diff) + 3 substitution + 17 positional + 8 structural |
| **combined** | Domain | ~65 | Engineered + Noah's thesis, deduplicated 18 overlaps |
| **ordinal** | Data | 3-4 | Position + integer AAs + species |
| **onehot** | Data | ~43 | Position + one-hot AAs (20×2) + species |
| **fullseq** | Data | ~500-900 | Full mutant sequence as integer vector |
| **noah_original** | Domain | 31 | Noah Plingen's thesis features (10 AA scales, MSA-aligned) |

In [ ]:
# Demo all encoding strategies
encoding_results = {}

for enc_name in ["ordinal", "onehot", "engineered", "combined"]:
    try:
        # Use cached extraction (cache-first, computes on miss)
        from scripts.run_experiment import encode_features_cached
        X_enc, fnames = encode_features_cached(df, enc_name, sequences)
        encoding_results[enc_name] = {"X": X_enc, "features": fnames}
        print(f"{enc_name:15s}: {X_enc.shape[1]:5d} features, {X_enc.shape[0]} samples")
    except Exception as e:
        print(f"{enc_name:15s}: SKIPPED — {e}")

print(f"\nEncoding comparison complete — {len(encoding_results)} strategies ready")

## 4. Feature Engineering Deep Dive

### Engineered Features (52 total)

| Group | Count | Features |
|-------|-------|----------|
| **Physicochemical** | 24 | 8 AAIndex scales × {wildtype, mutant, delta} |
| **Substitution** | 3 | BLOSUM62 (raw + normalized) + Grantham distance |
| **Positional** | 17 | Normalized position + 16 subunit one-hot |
| **Structural** | 8 | RSA, B-factor, DSSP×3, C-beta density, HSE×2 |

### nAChR-specific advantages over ENaC:
- **Grantham distance** — composite physicochemical distance (0-215), not in ENaC
- **Half-Sphere Exposure (HSE)** — up/down packing asymmetry, not in ENaC
- **Multi-PDB support** — 4 experimental + 6 AlphaFold structures covering 100% of subunits
- **Shrake-Rupley SASA** — pure Python, no `mkdssp` binary dependency

In [ ]:
# Inspect engineered features
encoder = NachrFeatureEncoder()
_ = encoder.fit_transform(df)
fnames = encoder.get_feature_names_out()

print(f"Total engineered features: {len(fnames)}")
for fg_name, fg_info in FEATURE_GROUPS.items():
    print(f"  {fg_name:20s}: {fg_info['n_features']:3d} features — {fg_info['desc']}")

print(f"\nAll feature names:\n" + "\n".join(f"  {i+1:2d}. {n}" for i, n in enumerate(fnames)))

## 5. Model Training — Nested Cross-Validation

**Strategy:** 5×5 nested CV with Optuna hyperparameter optimization (50 trials per fold), 5 random seeds for robust performance estimates.

**Models:** Logistic Regression, SVM (RBF), Random Forest, LightGBM, XGBoost, KNN, MLP, Gaussian NB

In [ ]:
# Quick evaluation (no Optuna, default hyperparameters) — fast sanity check
print("Quick evaluation (default HPs, no Optuna)...")
from scripts.run_experiment import run_quick_test, encode_features_cached

X_eng, fnames_eng = encode_features_cached(df, "engineered", sequences)
quick_results = run_quick_test(X_eng, labels, encoding="engineered")
print("\nQuick test complete.")

In [ ]:
# Full nested CV with Optuna (long-running — uncomment to run)
# result = nested_cross_validation(
#     X_eng, labels,
#     model_name="random_forest",
#     encoding="engineered",
#     n_outer_folds=5, n_inner_folds=5,
#     n_trials=50, n_jobs=-1, verbose=1,
#     feature_names=fnames_eng,
# )
# print(result.summary())
# result.save(RESULTS_DIR / "rf_engineered_full.json")

In [ ]:
# Aggregate and display existing results
summary = aggregate_results(RESULTS_DIR)
if not summary.empty:
    display(summary)
    # Plot comparison heatmap
    if "encoding" in summary.columns and summary["encoding"].nunique() > 1:
        plot_comparison_heatmap(summary)
        plt.show()
else:
    print("No results found in", RESULTS_DIR, "— run experiments first.")

## 6. Species Transfer Experiment

**Question:** Can mouse mutation data improve prediction of human mutation effects?

**Design:** Three training conditions evaluated on **identical human test folds**:
1. **Human-only** — baseline (within-species)
2. **Mouse-only** — cross-species transfer (tests evolutionary conservation)
3. **Mixed (human + mouse)** — augmentation hypothesis (tests if more data helps)

Paired Wilcoxon signed-rank test determines statistical significance.

In [ ]:
# Species transfer demo (requires multi-species data — uncomment when ready)
# from scripts.run_experiment import encode_features_cached
#
# df_multi, labels_multi, seqs_multi = load_multi_species_dataset(
#     mouse_path="path/to/mouse_data.xlsx"
# )
#
# X_st, fnames_st = encode_features_cached(df_multi, "engineered", seqs_multi)
# species_arr = df_multi["species"].values
#
# st_result = species_transfer_cv(
#     X_st, labels_multi, species_arr,
#     model_name="random_forest",
#     encoding="engineered",
#     n_outer_folds=5, n_inner_folds=5,
#     n_trials=30, n_jobs=-1, verbose=1,
# )
#
# print(st_result.summary())
# st_result.save(RESULTS_DIR / "species_transfer_rf_engineered.json")
#
# # Statistical test: does mixed significantly outperform human-only?
# from vep_nachr.training.evaluation import paired_comparison_test
# test_result = paired_comparison_test(
#     st_result.human_only_per_seed_f1,
#     st_result.mixed_per_seed_f1,
#     test="wilcoxon"
# )
# print(f"\nWilcoxon test: p={test_result['p_value']:.4f}, significant={test_result['significant_at_05']}")
#
# # Plot
# plot_species_transfer_comparison(
#     st_result.human_only_per_seed_f1,
#     st_result.mouse_only_per_seed_f1,
#     st_result.mixed_per_seed_f1,
#     output_path=RESULTS_DIR / "species_transfer_boxplot.png",
# )

## 7. Ablation Studies

Two modes determine which features drive prediction:

1. **Per-feature ablation** — drop each feature individually, measure F1 drop
2. **Per-group ablation** — leave-one-group-out (physicochemical, substitution, positional, structural)

Run via CLI: `python scripts/experiments/run_ablation.py --model random_forest --all`

In [ ]:
# Quick per-group ablation demo (single model, single seed)
from scripts.experiments.run_ablation import (
    run_baseline, run_per_group_ablation, ENGINEERED_GROUP_MAP
)

# Note: full per-feature ablation takes ~52× longer — use the CLI for that
X_eng, fnames_eng = encode_features_cached(df, "engineered", sequences)

# Baseline
print("Computing baseline...")
baseline_f1, baseline_std = run_baseline(X_eng, labels, "random_forest", n_trials=20, verbose=1)
print(f"Baseline F1: {baseline_f1:.4f} ± {baseline_std:.4f}\n")

# Per-group ablation
print("Running per-group ablation...")
group_results = run_per_group_ablation(
    X_eng, labels, fnames_eng, "random_forest",
    ENGINEERED_GROUP_MAP, baseline_f1,
    n_trials=20, n_jobs=-1, verbose=1,
)

print("\nGroup Importance Ranking:")
for i, r in enumerate(group_results):
    print(f"  {i+1}. {r.dropped:25s} F1 drop: {r.f1_drop:+.4f} (F1={r.mean_f1:.4f})")

## 8. Evaluation & Visualization

### Metrics computed:
- F1-score (binary), Precision, Recall, Accuracy
- Per-class F1 (LOF / GOF)
- ROC-AUC (when probabilities available)
- Confusion matrix
- Paired Wilcoxon/t-test for model comparison

### Feature importance:
- **SHAP** TreeExplainer for tree-based models (RF, LightGBM, XGBoost)
- **Permutation importance** for non-tree models
- Aggregated across CV folds with mean ± std

In [ ]:
# Compute metrics for an existing result
import json
result_files = sorted(RESULTS_DIR.glob("*.json"))
print(f"Found {len(result_files)} result files:")
for rf in result_files[:10]:
    print(f"  {rf.name}")

# If results exist, display the best one
summary = aggregate_results(RESULTS_DIR)
if not summary.empty:
    best = summary.iloc[0]
    print(f"\nBest result: {best['model']} × {best.get('encoding','?')}")
    print(f"  F1: {best['mean_f1']:.4f} ± {best['std_f1']:.4f}")
    print(f"  Accuracy: {best.get('mean_accuracy', 'N/A')}")

### SHAP Feature Importance

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions. We use TreeExplainer for tree-based models (Random Forest, LightGBM, XGBoost) and aggregate across CV folds.

In [ ]:
# SHAP demo — requires a trained model (uncomment when you have one)
# from sklearn.model_selection import train_test_split
#
# X_train, X_test, y_train, y_test = train_test_split(X_eng, labels, test_size=0.2, random_state=42)
#
# model = get_model("random_forest", random_state=42)
# model.fit(X_train, y_train)
#
# analyzer = FeatureImportanceAnalyzer(method="shap")
# analyzer.compute_importance(model, X_test, feature_names=fnames_eng)
# fi_result = analyzer.aggregate(model_name="random_forest", encoding="engineered")
#
# # Plot top-20 features
# plot_feature_importance(fi_result, top_n=20,
#                        output_path=RESULTS_DIR / "shap_importance.png")
# print("SHAP plot saved to results/")
# print(fi_result.to_dataframe().head(10))

## 9. Reproducibility

### Config YAML

Full experiment configurations can be saved/loaded as YAML for reproducible runs:

```python
from vep_nachr.config import Config, save_config, load_config

config = Config(
    experiment_name="species_transfer_v1",
    species=["human", "mouse"],
    models=["random_forest", "lightgbm"],
    encodings=["engineered", "ordinal", "onehot"],
)
config.to_yaml("experiment_config.yaml")
# Later: config = Config.from_yaml("experiment_config.yaml")
```

### Feature Caching

Feature matrices are automatically cached to `results/.cache/` to avoid re-extracting the same encoding for multiple models:

```python
from scripts.run_experiment import encode_features_cached
X, fnames = encode_features_cached(df, "engineered", sequences)  # cache-first
```

Cache keys are SHA-256 hashes of (encoding + mutation identities), so the cache auto-invalidates when data changes.

### Random Seeds

All experiments use fixed seeds `[42, 123, 456, 789, 1011]` for reproducible CV splits and model initialization.

## 10. CLI Quick Reference

```bash
# Quick test (no Optuna)
python scripts/run_experiment.py --quick

# Single model, engineered encoding
python scripts/run_experiment.py --model random_forest

# All encodings across all core models
python scripts/run_experiment.py --all-encodings

# Multi-species comparison
python scripts/run_experiment.py --mouse-data mouse_data.xlsx --encoding engineered

# Ablation studies
python scripts/experiments/run_ablation.py --model random_forest --all

# Species transfer (requires mouse data)
python scripts/run_experiment.py --mouse-data mouse.xlsx --encoding engineered

# Save config for reproducibility
python -c "from vep_nachr.config import *; Config(experiment_name='my_exp').to_yaml('exp.yaml')"
```

## 11. Package Architecture

```
VEP Nachr/
├── vep_nachr/                    # Python package
│   ├── config.py                 # Global config, FEATURE_GROUPS, Config (YAML)
│   ├── data/
│   │   ├── loader.py             # load_dataset(), load_multi_species_dataset()
│   │   └── encoders.py           # Ordinal/OneHot/FullSequence/get_encoder()
│   ├── features/
│   │   ├── encoder.py            # NachrFeatureEncoder (52 features)
│   │   ├── physicochemical.py    # 8 AAIndex scales × {wt, mt, diff}
│   │   ├── substitution.py       # BLOSUM62 + Grantham distance
│   │   ├── structural.py         # Multi-PDB: RSA, B-factor, DSSP, HSE, C-beta
│   │   ├── noah_original.py      # NoahOriginalFeatureEncoder + Combined
│   │   └── noah_features/        # Noah's thesis subpackage (6 files)
│   ├── models/
│   │   └── registry.py           # 9 models + Optuna HP spaces
│   └── training/
│       ├── cross_validation.py   # nested_cv, species_transfer_cv
│       └── evaluation.py         # Metrics, SHAP, plots, stats
├── scripts/
│   ├── run_experiment.py         # Main experiment runner + feature caching
│   ├── experiments/
│   │   └── run_ablation.py       # Per-feature + per-group ablation
│   └── download_alphafold_structures.py
├── data/
│   └── raw/structure_files/       # 10 PDB/CIF structures
├── results/                       # JSON results + .cache/
├── MASTER.ipynb                   # This notebook
└── requirements.txt
```

## 12. Remaining Work (post-Task-8)

- [ ] Paper generation scripts (port from ENaC's `scripts/paper/`)
- [ ] HPC/SLURM job scripts (port from ENaC's `scripts/hpc/`)
- [ ] CatBoost subprocess isolation (prevent C++ segfaults)
- [ ] Pseudo-resolution for terminal IDR positions
- [ ] Generate MSA file for aligned position features